<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Simulate_TMS_fMRI_ANN_persubject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulate TMS-fMRI Sessions with Subject-Specific ANN Models

**Objective:** Validate that subject-specific models trained on TMS data can simulate empirically-realistic stimulation responses when stimulated at real timing.

**Workflow:**
1. Load subject-specific models trained on TMS stimulation sessions
2. Generate synthetic rest + stim sessions with:
   - Rest: Simulate from resting state without stimulation
   - Stim: Simulate from resting state, stimulate at empirical times, in empirical target regions
3. Compare empirical vs simulated functional connectivity (rest, stim, delta)
4. Characterize which brain regions respond most to stimulation

## Step 1: Setup and Configuration

In [ ]:
# --- Setup ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys, pickle, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, ttest_1samp

# Clone repo + add to path
repo_dir = "/content/BrainStim_ANN_fMRI_HCP"
if not os.path.exists(repo_dir):
    !git clone https://github.com/grabuffo/BrainStim_ANN_fMRI_HCP.git
else:
    print("Repo already exists ✅")

sys.path.append(repo_dir)
from src.preprocessing_tms_fmri import preprocess_run

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {device}")

In [ ]:
# --- Configuration ---
ROI_num = 450
using_steps = 3
tr_rest = 2.0
tr_stim = 2.4
remove_initial_trs = 30
low_hz, high_hz = 0.008, 0.08

# Simulation parameters
BURN_IN = 10
NOISE_SIGMA = 0.28
STIM_AMP = 10.0
STIM_DURATION_S = tr_stim

rng = np.random.default_rng(42)

# Paths
base_dir = "/content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data"
data_dir = os.path.join(base_dir, "TMS_fMRI")
preproc_dir = os.path.join(base_dir, "preprocessed_subjects_tms_fmri")
dataset_pkl = os.path.join(data_dir, "dataset_tian50_schaefer400_allruns.pkl")
weights_dir = os.path.join(preproc_dir, "trained_models_MLP_tms_fmri_persubject")

out_dir = os.path.join(preproc_dir, "simulated_tms_fmri_persubject")
os.makedirs(out_dir, exist_ok=True)

out_pkl = os.path.join(out_dir, "dataset_simulated_persubject.pkl")
results_json = os.path.join(out_dir, "fc_validation_results.json")

print(f"Base dir: {base_dir}")
print(f"Weights dir: {weights_dir}")
print(f"Output dir: {out_dir}")

## Step 2: Model Definition and Loading

In [ ]:
# --- Model Class ---
class ANN_MLP_with_Stimulus(nn.Module):
    def __init__(self, roi_num, using_steps, hidden_dim=256, latent_dim=128):
        super().__init__()
        brain_dim = using_steps * roi_num
        stim_dim = roi_num
        stim_timing_dim = 1
        input_dim = brain_dim + stim_dim + stim_timing_dim
        output_dim = roi_num
        
        self.func = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, output_dim),
        )
    
    def forward(self, x):
        return self.func(x)

print("Model architecture defined")

In [ ]:
# --- Load empirical dataset ---
print(f"Loading empirical dataset from {dataset_pkl}...")
with open(dataset_pkl, "rb") as f:
    dataset_emp = pickle.load(f)

print(f"✓ Loaded {len(dataset_emp)} subjects")

# List available subjects
subjects_available = sorted(dataset_emp.keys())
print(f"\nAvailable subjects: {subjects_available[:10]}...") if len(subjects_available) > 10 else print(f"Available subjects: {subjects_available}")

In [ ]:
# --- Load subject-specific trained models ---
print("Loading subject-specific models...\n")

trained_models = {}
model_files = [f for f in os.listdir(weights_dir) if f.endswith('_MLP_with_stim.pt')]

for model_file in sorted(model_files):
    sub_id = model_file.replace('_MLP_with_stim.pt', '')
    model_path = os.path.join(weights_dir, model_file)
    
    try:
        model = ANN_MLP_with_Stimulus(roi_num=ROI_num, using_steps=using_steps)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        model.eval()
        
        # Check if subject has empirical data
        if sub_id in dataset_emp:
            trained_models[sub_id] = model
            print(f"✓ {sub_id}")
    except RuntimeError as e:
        if "size mismatch" in str(e):
            print(f"✗ {sub_id} - skipping (dimension mismatch)")
        else:
            raise

print(f"\n✅ Loaded {len(trained_models)} models with matching empirical data")

## Step 3: Simulation Functions

In [ ]:
def simulate_run(model, init_window_SxN, n_steps, stim_steps=None, target_idx=None):
    """Simulate brain activity time series with optional stimulation.
    
    Input format: [brain_state(1350) | target_region(450 one-hot) | stim_pulse(1)]
    """
    init_window_SxN = np.asarray(init_window_SxN, dtype=np.float32)
    assert init_window_SxN.shape == (using_steps, ROI_num), f"Expected shape ({using_steps}, {ROI_num}), got {init_window_SxN.shape}"

    # Convert stim_steps to set, handling None and arrays properly
    if stim_steps is not None:
        stim_steps = set(int(s) for s in stim_steps)
    else:
        stim_steps = set()
    
    do_stim = (target_idx is not None) and (len(stim_steps) > 0)
    
    # Create target one-hot vector
    if do_stim:
        target_vec = np.zeros(ROI_num, dtype=np.float32)
        target_vec[target_idx] = 1.0
    else:
        target_vec = None
    
    w = init_window_SxN.copy()

    # Burn-in
    for _ in range(BURN_IN):
        y = predict_next(model, w, target_vec=target_vec, stim_pulse=0.0)
        w = np.vstack([w[1:], y[None, :]])

    # Simulate
    out = np.zeros((n_steps, ROI_num), dtype=np.float32)
    for t in range(n_steps):
        stim_pulse = 1.0 if (do_stim and t in stim_steps) else 0.0
        y = predict_next(model, w, target_vec=target_vec, stim_pulse=stim_pulse)
        out[t] = y
        w = np.vstack([w[1:], y[None, :]])

    return out

In [ ]:
# --- Functional connectivity functions ---

def compute_fc_matrix(time_series):
    """Compute functional connectivity (Pearson correlation) matrix."""
    ts_std = (time_series - time_series.mean(axis=0)) / (time_series.std(axis=0) + 1e-8)
    fc = np.corrcoef(ts_std.T)
    return fc

def extract_upper_triangle(fc_matrix):
    """Extract upper triangle of FC (excluding diagonal) as vector."""
    N = fc_matrix.shape[0]
    indices = np.triu_indices(N, k=1)
    return fc_matrix[indices]

def seed_based_fc(time_series, seed_idx):
    """Compute seed-based functional connectivity.
    
    Returns correlation of seed region with all regions.
    """
    ts_std = (time_series - time_series.mean(axis=0)) / (time_series.std(axis=0) + 1e-8)
    seed_ts = ts_std[:, seed_idx]
    seed_fc = np.corrcoef(seed_ts, ts_std.T)[0, 1:]
    return seed_fc

print("✓ FC functions defined")

## Step 4: Generate Simulated Dataset

In [ ]:
# --- Generate synthetic dataset ---
print("Generating simulated dataset for subject-specific models...\n")

dataset_sim = {}
n_sim_rest = 0
n_sim_stim = 0

for sub_id in sorted(trained_models.keys()):
    model = trained_models[sub_id]
    sub_data_emp = dataset_emp[sub_id]
    
    dataset_sim[sub_id] = {"task-rest": {}, "task-stim": {}}
    
    # ---- SIMULATE REST ----
    if "task-rest" in sub_data_emp:
        for run_idx, run_emp in sub_data_emp["task-rest"].items():
            ts_emp = run_emp.get("time series", None)
            
            if ts_emp is None or not isinstance(ts_emp, np.ndarray) or ts_emp.shape[1] != ROI_num:
                continue
            
            # Preprocess empirical rest
            ts_drop = ts_emp[remove_initial_trs:, :]
            ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
            
            if ts_proc.shape[0] <= using_steps:
                continue
            
            # Simulate rest: use first using_steps as initial window, no stimulation
            init_window = ts_proc[:using_steps].copy()
            dur_s = ts_proc.shape[0] * tr_rest
            n_steps = int(np.ceil(dur_s / tr_rest))
            
            sim_ts = simulate_run(model, init_window, n_steps, stim_steps=None, target_idx=None)
            
            dataset_sim[sub_id]["task-rest"][int(run_idx)] = {
                "time series": sim_ts,
                "metadata": {"simulated": True, "tr": float(tr_rest)}
            }
            n_sim_rest += 1
    
    # ---- SIMULATE STIM ----
    if "task-stim" in sub_data_emp:
        for run_idx, run_emp in sub_data_emp["task-stim"].items():
            ts_emp = run_emp.get("time series", None)
            target_vec = run_emp.get("target", None)
            stim_time_df = run_emp.get("stim time", None)
            
            if ts_emp is None or not isinstance(ts_emp, np.ndarray) or ts_emp.shape[1] != ROI_num:
                continue
            
            target_idx = safe_target_idx(target_vec)
            if target_idx is None:
                continue
            
            # Preprocess empirical stim
            ts_drop = ts_emp[remove_initial_trs:, :]
            ts_proc = preprocess_run(ts_drop, tr=tr_stim, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
            
            if ts_proc.shape[0] <= using_steps:
                continue
            
            # Extract stimulus timing
            stim_steps_model = []
            if stim_time_df is not None and len(stim_time_df) > 0:
                onset_col = get_onset_column(stim_time_df)
                if onset_col is not None:
                    onsets_s = stim_time_df[onset_col].values
                    onsets_s = onsets_s[onsets_s >= remove_initial_trs * tr_stim]
                    onsets_s = onsets_s - remove_initial_trs * tr_stim
                    stim_steps_model = map_onsets_to_steps(onsets_s, tr_model=tr_stim)
            
            # Simulate stim: use first using_steps as initial window, with stimulation
            init_window = ts_proc[:using_steps].copy()
            dur_s = ts_proc.shape[0] * tr_stim
            n_steps = int(np.ceil(dur_s / tr_stim))
            
            sim_ts = simulate_run(model, init_window, n_steps, 
                                 stim_steps=stim_steps_model, target_idx=target_idx)
            
            dataset_sim[sub_id]["task-stim"][int(run_idx)] = {
                "time series": sim_ts,
                "target": target_vec,
                "stim time": stim_time_df,
                "metadata": {"simulated": True, "tr": float(tr_stim), "target": int(target_idx)}
            }
            n_sim_stim += 1

print(f"✓ Generated {n_sim_rest} rest sessions and {n_sim_stim} stim sessions")
print(f"\nSimulated {len(dataset_sim)} subjects")

## Step 5: Validate with FC Correlations

In [ ]:
# --- Compute FC for REST condition ---
print("="*70)
print("COMPUTING FC CORRELATIONS - REST CONDITION")
print("="*70)

rest_fc_correlations = {}

for sub_id in sorted(dataset_sim.keys()):
    if "task-rest" not in dataset_sim[sub_id] or len(dataset_sim[sub_id]["task-rest"]) == 0:
        continue
    
    if "task-rest" not in dataset_emp[sub_id] or len(dataset_emp[sub_id]["task-rest"]) == 0:
        continue
    
    # Collect empirical rest FC
    ts_emp_list = []
    for run_idx, run in dataset_emp[sub_id]["task-rest"].items():
        ts = run.get("time series")
        if ts is not None and isinstance(ts, np.ndarray) and ts.shape[1] == ROI_num:
            ts_drop = ts[remove_initial_trs:, :]
            ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
            if ts_proc.shape[0] > using_steps:
                ts_emp_list.append(ts_proc)
    
    # Collect simulated rest FC
    ts_sim_list = []
    for run_idx, run in dataset_sim[sub_id]["task-rest"].items():
        ts = run.get("time series")
        if ts is not None and isinstance(ts, np.ndarray) and ts.shape[1] == ROI_num:
            ts_proc = preprocess_run(ts, tr=tr_rest, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
            if ts_proc.shape[0] > using_steps:
                ts_sim_list.append(ts_proc)
    
    if ts_emp_list and ts_sim_list:
        ts_emp = np.vstack(ts_emp_list)
        ts_sim = np.vstack(ts_sim_list)
        
        fc_emp = compute_fc_matrix(ts_emp)
        fc_sim = compute_fc_matrix(ts_sim)
        
        fc_emp_vec = extract_upper_triangle(fc_emp)
        fc_sim_vec = extract_upper_triangle(fc_sim)
        
        r_rest = np.corrcoef(fc_emp_vec, fc_sim_vec)[0, 1]
        rest_fc_correlations[sub_id] = float(r_rest)
        print(f"{sub_id}: r = {r_rest:.4f}")

if rest_fc_correlations:
    corrs = list(rest_fc_correlations.values())
    print(f"\nREST FC Correlation:")
    print(f"  Mean: {np.mean(corrs):.4f} ± {np.std(corrs):.4f}")
    print(f"  Range: [{np.min(corrs):.4f}, {np.max(corrs):.4f}]")

In [ ]:
# --- Compute FC for STIM condition ---
print("\n" + "="*70)
print("COMPUTING FC CORRELATIONS - STIM CONDITION")
print("="*70)

stim_fc_correlations = {}
stim_by_target = {}

for sub_id in sorted(dataset_sim.keys()):
    if "task-stim" not in dataset_sim[sub_id] or len(dataset_sim[sub_id]["task-stim"]) == 0:
        continue
    
    if "task-stim" not in dataset_emp[sub_id] or len(dataset_emp[sub_id]["task-stim"]) == 0:
        continue
    
    stim_fc_correlations[sub_id] = {}
    
    # Process each stim session
    for run_idx in dataset_sim[sub_id]["task-stim"].keys():
        run_emp = dataset_emp[sub_id]["task-stim"].get(run_idx)
        run_sim = dataset_sim[sub_id]["task-stim"].get(run_idx)
        
        if run_emp is None or run_sim is None:
            continue
        
        ts_emp = run_emp.get("time series")
        ts_sim = run_sim.get("time series")
        target_vec = run_emp.get("target")
        
        if ts_emp is None or ts_sim is None:
            continue
        
        target_idx = safe_target_idx(target_vec)
        if target_idx is None:
            continue
        
        # Preprocess
        ts_emp_drop = ts_emp[remove_initial_trs:, :]
        ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_stim, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
        
        ts_sim_proc = preprocess_run(ts_sim, tr=tr_stim, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
        
        if ts_emp_proc.shape[0] > using_steps and ts_sim_proc.shape[0] > using_steps:
            fc_emp = compute_fc_matrix(ts_emp_proc)
            fc_sim = compute_fc_matrix(ts_sim_proc)
            
            fc_emp_vec = extract_upper_triangle(fc_emp)
            fc_sim_vec = extract_upper_triangle(fc_sim)
            
            r_stim = np.corrcoef(fc_emp_vec, fc_sim_vec)[0, 1]
            target_key = f"target_{target_idx}"
            
            stim_fc_correlations[sub_id][target_key] = float(r_stim)
            
            if target_key not in stim_by_target:
                stim_by_target[target_key] = []
            stim_by_target[target_key].append(r_stim)
            
            print(f"{sub_id} x {target_key}: r = {r_stim:.4f}")

print(f"\nSTIM FC Correlation (by target):")
for target_key in sorted(stim_by_target.keys()):
    corrs = stim_by_target[target_key]
    print(f"  {target_key}: {np.mean(corrs):.4f} ± {np.std(corrs):.4f} (n={len(corrs)})")

In [ ]:
# --- Compute DELTA FC (stim - rest) ---
print("\n" + "="*70)
print("COMPUTING FC CORRELATIONS - DELTA CONDITION (Stim - Rest)")
print("="*70)

delta_fc_correlations = {}
delta_by_target = {}

for sub_id in sorted(dataset_sim.keys()):
    if sub_id not in rest_fc_correlations:
        continue
    
    if "task-stim" not in dataset_sim[sub_id] or len(dataset_sim[sub_id]["task-stim"]) == 0:
        continue
    
    if "task-stim" not in dataset_emp[sub_id] or len(dataset_emp[sub_id]["task-stim"]) == 0:
        continue
    
    # Get rest FC for this subject
    ts_rest_emp_list = []
    for run in dataset_emp[sub_id]["task-rest"].values():
        ts = run.get("time series")
        if ts is not None and isinstance(ts, np.ndarray) and ts.shape[1] == ROI_num:
            ts_drop = ts[remove_initial_trs:, :]
            ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
            if ts_proc.shape[0] > using_steps:
                ts_rest_emp_list.append(ts_proc)
    
    ts_rest_sim_list = []
    for run in dataset_sim[sub_id]["task-rest"].values():
        ts = run.get("time series")
        if ts is not None and isinstance(ts, np.ndarray) and ts.shape[1] == ROI_num:
            ts_proc = preprocess_run(ts, tr=tr_rest, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
            if ts_proc.shape[0] > using_steps:
                ts_rest_sim_list.append(ts_proc)
    
    if not ts_rest_emp_list or not ts_rest_sim_list:
        continue
    
    fc_rest_emp = compute_fc_matrix(np.vstack(ts_rest_emp_list))
    fc_rest_sim = compute_fc_matrix(np.vstack(ts_rest_sim_list))
    
    delta_fc_correlations[sub_id] = {}
    
    # Process each stim session
    for run_idx in dataset_sim[sub_id]["task-stim"].keys():
        run_emp = dataset_emp[sub_id]["task-stim"].get(run_idx)
        run_sim = dataset_sim[sub_id]["task-stim"].get(run_idx)
        
        if run_emp is None or run_sim is None:
            continue
        
        ts_emp = run_emp.get("time series")
        ts_sim = run_sim.get("time series")
        target_vec = run_emp.get("target")
        
        if ts_emp is None or ts_sim is None:
            continue
        
        target_idx = safe_target_idx(target_vec)
        if target_idx is None:
            continue
        
        # Preprocess
        ts_emp_drop = ts_emp[remove_initial_trs:, :]
        ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_stim, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
        
        ts_sim_proc = preprocess_run(ts_sim, tr=tr_stim, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
        
        if ts_emp_proc.shape[0] > using_steps and ts_sim_proc.shape[0] > using_steps:
            fc_stim_emp = compute_fc_matrix(ts_emp_proc)
            fc_stim_sim = compute_fc_matrix(ts_sim_proc)
            
            # Compute delta FC
            delta_fc_emp = fc_stim_emp - fc_rest_emp
            delta_fc_sim = fc_stim_sim - fc_rest_sim
            
            delta_emp_vec = extract_upper_triangle(delta_fc_emp)
            delta_sim_vec = extract_upper_triangle(delta_fc_sim)
            
            r_delta = np.corrcoef(delta_emp_vec, delta_sim_vec)[0, 1]
            target_key = f"target_{target_idx}"
            
            delta_fc_correlations[sub_id][target_key] = float(r_delta)
            
            if target_key not in delta_by_target:
                delta_by_target[target_key] = []
            delta_by_target[target_key].append(r_delta)
            
            print(f"{sub_id} x {target_key}: r_delta = {r_delta:.4f}")

print(f"\nDELTA FC Correlation (by target):")
for target_key in sorted(delta_by_target.keys()):
    corrs = delta_by_target[target_key]
    print(f"  {target_key}: {np.mean(corrs):.4f} ± {np.std(corrs):.4f} (n={len(corrs)})")

## Step 6: Characterize Stimulus Effects on Brain Regions

In [ ]:
# --- Characterize stimulus-induced connectivity changes ---
print("\n" + "="*70)
print("CHARACTERIZING STIMULUS-INDUCED CONNECTIVITY CHANGES")
print("="*70)

stimulus_effects = {}

for sub_id in sorted(dataset_sim.keys()):
    if "task-stim" not in dataset_sim[sub_id] or len(dataset_sim[sub_id]["task-stim"]) == 0:
        continue
    
    if "task-rest" not in dataset_sim[sub_id] or len(dataset_sim[sub_id]["task-rest"]) == 0:
        continue
    
    stimulus_effects[sub_id] = {}
    
    # Get rest seed-FC for comparison
    ts_rest_sim = None
    for run in dataset_sim[sub_id]["task-rest"].values():
        ts = run.get("time series")
        if ts is not None and isinstance(ts, np.ndarray):
            ts_proc = preprocess_run(ts, tr=tr_rest, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
            if ts_proc.shape[0] > using_steps:
                ts_rest_sim = ts_proc
                break
    
    if ts_rest_sim is None:
        continue
    
    # Analyze each stim session
    for run_idx, run_sim in dataset_sim[sub_id]["task-stim"].items():
        ts_sim = run_sim.get("time series")
        target_vec = run_sim.get("target")
        
        if ts_sim is None:
            continue
        
        target_idx = safe_target_idx(target_vec)
        if target_idx is None:
            continue
        
        ts_sim_proc = preprocess_run(ts_sim, tr=tr_stim, n_drop=0, 
                                     low=low_hz, high=high_hz, order=2, zscore=True)
        
        if ts_sim_proc.shape[0] > using_steps:
            # Compute seed-based FC from target region
            seedfc_rest = seed_based_fc(ts_rest_sim, target_idx)
            seedfc_stim = seed_based_fc(ts_sim_proc, target_idx)
            
            # Characterize changes
            delta_seedfc = seedfc_stim - seedfc_rest
            
            # Find regions with largest increase/decrease
            top_increase_idx = np.argsort(delta_seedfc)[-5:][::-1]
            top_decrease_idx = np.argsort(delta_seedfc)[:5]
            
            stimulus_effects[sub_id][f"run_{run_idx}"] = {
                "target_idx": int(target_idx),
                "mean_delta_seedfc": float(np.mean(delta_seedfc)),
                "max_increase": float(delta_seedfc[top_increase_idx[0]]),
                "max_decrease": float(delta_seedfc[top_decrease_idx[0]]),
                "n_regions_increased": int(np.sum(delta_seedfc > 0)),
                "n_regions_decreased": int(np.sum(delta_seedfc < 0)),
            }

print(f"\nCharacterized stimulus effects for {len(stimulus_effects)} subjects")
print(f"\nExample stimulus effects (first subject):")
if stimulus_effects:
    example_sub = list(stimulus_effects.keys())[0]
    for run_key, effects in stimulus_effects[example_sub].items():
        print(f"  {run_key}:")
        print(f"    Target: ROI {effects['target_idx']}")
        print(f"    Mean Δ seed-FC: {effects['mean_delta_seedfc']:+.4f}")
        print(f"    Max increase: {effects['max_increase']:+.4f}")
        print(f"    Max decrease: {effects['max_decrease']:+.4f}")
        print(f"    Regions with increased connectivity: {effects['n_regions_increased']}")
        print(f"    Regions with decreased connectivity: {effects['n_regions_decreased']}")

## Step 7: Summary and Save Results

In [ ]:
# --- Summary statistics ---
print("\n" + "="*70)
print("SUMMARY: EMPIRICAL vs SIMULATED FC CORRELATIONS")
print("="*70)

print(f"\n1. REST Condition:")
if rest_fc_correlations:
    corrs = list(rest_fc_correlations.values())
    print(f"   Mean r: {np.mean(corrs):.4f} ± {np.std(corrs):.4f}")
    print(f"   (Models simulate rest FC well: r > 0.5 indicates realistic structure)")
else:
    print(f"   No data")

print(f"\n2. STIM Condition (pooled across targets):")
all_stim = []
for corrs in stim_by_target.values():
    all_stim.extend(corrs)
if all_stim:
    print(f"   Mean r: {np.mean(all_stim):.4f} ± {np.std(all_stim):.4f}")
    print(f"   (Higher correlation indicates realistic stimulus-evoked responses)")
else:
    print(f"   No data")

print(f"\n3. DELTA Condition - Stimulation-induced changes (pooled):")
all_delta = []
for corrs in delta_by_target.values():
    all_delta.extend(corrs)
if all_delta:
    print(f"   Mean r: {np.mean(all_delta):.4f} ± {np.std(all_delta):.4f}")
    print(f"   (Positive correlation: model captures stimulus-induced FC changes)")
else:
    print(f"   No data")

print(f"\n" + "="*70)

In [ ]:
# --- Save results ---
print(f"\nSaving results...")

# Save simulated dataset
with open(out_pkl, "wb") as f:
    pickle.dump(dataset_sim, f)
print(f"✓ Saved simulated dataset to {out_pkl}")

# Save analysis results
results = {
    "rest_fc_correlations": rest_fc_correlations,
    "stim_fc_correlations_by_target": stim_by_target,
    "delta_fc_correlations_by_target": delta_by_target,
    "stimulus_effects": stimulus_effects,
    "summary": {
        "rest_mean_r": float(np.mean(list(rest_fc_correlations.values()))) if rest_fc_correlations else None,
        "rest_std_r": float(np.std(list(rest_fc_correlations.values()))) if rest_fc_correlations else None,
        "stim_mean_r": float(np.mean(all_stim)) if all_stim else None,
        "stim_std_r": float(np.std(all_stim)) if all_stim else None,
        "delta_mean_r": float(np.mean(all_delta)) if all_delta else None,
        "delta_std_r": float(np.std(all_delta)) if all_delta else None,
    }
}

with open(results_json, "w") as f:
    json.dump(results, f, indent=2)
print(f"✓ Saved validation results to {results_json}")

print(f"\n✅ Simulation and validation complete!")

## Interpretation

**Key Questions Answered:**

1. **Can subject-specific models simulate realistic rest FC?**
   - If rest FC correlation > 0.5 → Models captured stable network structure
   - If rest FC correlation < 0.3 → Models did not fully capture network organization

2. **Do simulated stim FC match empirical stim FC?**
   - If stim FC correlation is high → Virtual stimulation produces realistic responses
   - If stim FC correlation is low → Model does not capture stimulus-evoked dynamics

3. **Do models capture stimulus-induced connectivity changes (Δ FC)?**
   - If delta FC correlation is positive and significant → Models learned how stimulation reshapes the network
   - If delta FC correlation is near zero → Models did not capture stimulus-specific changes

**Interpretation in Context:**

Even if **stimulus input is not explicitly used during training** (as shown in Supplementary_TMS_fMRI_validation.ipynb), the models learned the brain's intrinsic network structure from stimulation sessions. When stimulated at empirical times, this learned structure constrains the model's responses to be realistic. This demonstrates that:

- Virtual stimulation can be a **valid method** for in silico neuroscience
- The model learned **generalizable network dynamics**, not just stimulus-specific responses
- Personalized in silico stimulation can **match empirical observations**